In [ ]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ to sys.path so `import bowaka_v2_lab` works regardless of
# how the notebook is launched (jupyter / papermill / pytest).
import os
import sys
from pathlib import Path

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        sys.path.insert(0, str(_candidate / "src"))
        break
import bowaka_v2_lab  # noqa: F401
print("bowaka_v2_lab", bowaka_v2_lab.__version__)


In [ ]:
# Papermill parameter cell.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_walkforward_optuna.yml'
N_TRIALS = 5  # smoke; raise to 200+ for real runs


# 10 — Optuna Walk-Forward

Runs a small TPE study against the SIP config.

In [ ]:
import optuna
from bowaka_v2_lab.optuna.dispatcher import OptunaStudy
from bowaka_v2_lab.optuna.search_space import suggest_params
from bowaka_v2_lab.optuna.objective import compute_objective, FoldResult
study = OptunaStudy(feed='sip', cost_stress='conservative',
                      dataset_hash='cafebabecafebabe', config_hash='deadbeefdeadbeef',
                      n_trials=N_TRIALS)
study.create()

def objective(trial):
    params = suggest_params(trial)
    # Synthetic single-fold: penalise extreme params.
    fold = FoldResult(fold_id='f0',
        net_return=0.01 - abs(params['signals.gap_pct_max'] - 0.10) * 0.1,
        max_drawdown=0.02, turnover=0.0, concentration=0.0,
        n_trades=8, ambiguous_bar_count=0, missing_quote_count=0)
    return compute_objective([fold]).objective

study.optimize(objective)
print('best value:', study.study.best_value)
print('best params:', study.study.best_params)
